# Raw Olist Dataset Preparation

This notebook builds the merged Olist raw dataset from source CSV files and derives the weekly São Paulo product revenue dataset for 2017.

The final output is a weekly revenue dataset suitable for feature engineering, clustering, and temporal analysis.


## 1) Load source files

Read source orders, order items, customers, and products. Parse purchase timestamps immediately for correct year/week grouping.


In [35]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")
orders_path = data_dir / "olist_orders_dataset.csv"
customers_path = data_dir / "olist_customers_dataset.csv"
order_items_path = data_dir / "olist_order_items_dataset.csv"
products_path = data_dir / "olist_products_dataset.csv"

orders = pd.read_csv(orders_path, parse_dates=["order_purchase_timestamp"], low_memory=False)
customers = pd.read_csv(customers_path, low_memory=False)
products = pd.read_csv(products_path, low_memory=False)
print("Loaded orders rows:", len(orders))
print("Loaded customers rows:", len(customers))
print("Loaded products rows:", len(products))


Loaded orders rows: 99441
Loaded customers rows: 99441
Loaded products rows: 32951


In [36]:
# Quick sanity check of existing notebook variables
print("weekly_output_path:", weekly_output_path)
print("weekly_product_matrix_output:", weekly_product_matrix_output)

print("\nweekly_product_matrix:")
print(type(weekly_product_matrix))
print("shape:", weekly_product_matrix.shape)
display(weekly_product_matrix.head())

print("\nweekly_sp_product_revenue:")
print(type(weekly_sp_product_revenue))
print("shape:", weekly_sp_product_revenue.shape)
print("columns:", weekly_sp_product_revenue.columns.tolist())
display(weekly_sp_product_revenue.head())

weekly_output_path: ../data/SP_2017_sp_weekly_product_revenue.csv
weekly_product_matrix_output: ../data/SP_2017_weekly_product_revenue_by_product_id.csv

weekly_product_matrix:
<class 'pandas.core.frame.DataFrame'>
shape: (51, 4475)


product_id,00088930e925c41fd95ebfe695fd2655,0009406fd7479715e4bef61dd91f2462,00126f27c813603687e6ce486d909d01,001795ec6f1b187d37335e1c4704762e,001b72dfd63e9833e8c02742adf472e3,00210e41887c2a8ef9f791ebc780cc36,00250175f79f584c14ab5cecd80553cd,003c0b8f6580c850bd2e32044d2ac307,007c63ae4b346920756b5adcad8095de,008cff0e5792219fae03e570f980b330,...,ff922797a6771cab4e0c51d482285ec3,ff95ac47246ef13e48712ea1ff8df0d9,ffa1ce7f2a287ca5e369673bd77d43de,ffa4ff381e25872b5bf85353ff7ab0c9,ffb64e34a37740dafb6c88f1abd1fa61,ffb97eb64c6fe1baada2410288c04457,ffbbf6b9097237a1122f17e7341a3fb2,ffc48c754b5bd736e2887e279d1dec72,ffce5ed9e0bcc2e46796b988cdac733b,ffe8083298f95571b4a66bfbc1c05524
woy,,,,,,,,,,,,,,,,,,,,,
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



weekly_sp_product_revenue:
<class 'pandas.core.frame.DataFrame'>
shape: (15371, 4)
columns: ['year_week', 'product_id', 'product_category_name', 'weekly_revenue']


,year_week,product_id,product_category_name,weekly_revenue
0,2017-02,1a2d84c11fbc7b3d7c0a10206c085ce5,telefonia_fixa,10.49
1,2017-02,3889a02f55ae9a5e5e321c4a4d2eee64,moveis_decoracao,34.90
2,2017-02,743801b34cc44776de511ba8eff778e2,moveis_quarto,9.90
3,2017-02,a25a9e3433043fb8a0808f310d8d8203,fashion_bolsas_e_acessorios,129.99
4,2017-03,0ab80f38a21093b448518f68efe82c24,utilidades_domesticas,19.90


## 2) Merge orders with order items and customer geography

Join order-level data with item-level price and product IDs, then add customer destination state.


In [37]:
if not {"order_item_id", "product_id", "price"}.issubset(orders.columns):
    order_items = pd.read_csv(order_items_path, low_memory=False)
    orders = pd.merge(
        orders,
        order_items[["order_id", "order_item_id", "product_id", "price"]],
        on="order_id",
        how="left",
    )

merged_df = pd.merge(
    orders,
    customers[["customer_id", "customer_zip_code_prefix", "customer_city", "customer_state"]],
    on="customer_id",
    how="left",
)
raw_output_path = data_dir / "raw_olist_example_dataset.csv"
merged_df.to_csv(raw_output_path, index=False)
print("Wrote merged raw dataset to:", raw_output_path)


Wrote merged raw dataset to: ../data/raw_olist_example_dataset.csv


## 3) Scope to São Paulo and 2017

Filter the merged dataset to SP destination customers and orders placed in 2017.


In [38]:
merged_df["order_purchase_timestamp"] = pd.to_datetime(merged_df["order_purchase_timestamp"])
delivered_orders = merged_df[merged_df["order_status"] == "delivered"].copy()
sp_orders = delivered_orders[delivered_orders["customer_state"] == "SP"].copy()
sp_orders_2017 = sp_orders[sp_orders["order_purchase_timestamp"].dt.year == 2017].copy()
print("SP 2017 delivered rows:", len(sp_orders_2017))


SP 2017 delivered rows: 19513


## 4) Add product category and compute weekly revenue

Merge SP 2017 orders with product metadata and aggregate revenue by year/week, product, and category.


In [39]:
sp_products = pd.merge(
    sp_orders_2017,
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left",
)
sp_products["year_week"] = sp_products["order_purchase_timestamp"].dt.to_period("W").dt.strftime("%Y-%U")
weekly_sp_product_revenue = sp_products.groupby(
    ["year_week", "product_id", "product_category_name"],
    observed=True,
)["price"].sum().reset_index(name="weekly_revenue")

# Trim low-revenue inventory before creating the final weekly outputs
min_revenue_threshold = 200
product_total_revenue = (
    weekly_sp_product_revenue.groupby("product_id", observed=True)["weekly_revenue"]
    .sum()
    .reset_index()
)

high_revenue_products = product_total_revenue[
    product_total_revenue["weekly_revenue"] >= min_revenue_threshold
]["product_id"]

trimmed_weekly_revenue = weekly_sp_product_revenue[
    weekly_sp_product_revenue["product_id"].isin(high_revenue_products)
].copy()

weekly_output_path = data_dir / "SP_2017_sp_weekly_product_revenue.csv"
trimmed_weekly_revenue.to_csv(weekly_output_path, index=False)
print("Wrote weekly SP product revenue dataset to:", weekly_output_path)
print("Unique high-revenue products:", trimmed_weekly_revenue["product_id"].nunique())

pivot_output_path = data_dir / "SP_2017_sp_weekly_product_revenue_matrix.csv"
sp_revenue_matrix = trimmed_weekly_revenue.pivot_table(
    index="year_week",
    columns="product_category_name",
    values="weekly_revenue",
    aggfunc="sum",
    fill_value=0,
)
sp_revenue_matrix.to_csv(pivot_output_path)
print("Wrote weekly SP revenue pivot matrix to:", pivot_output_path)


Wrote weekly SP product revenue dataset to: ../data/SP_2017_sp_weekly_product_revenue.csv
Unique high-revenue products: 2451
Wrote weekly SP revenue pivot matrix to: ../data/SP_2017_sp_weekly_product_revenue_matrix.csv


## 5) Validate output

Check the final weekly revenue dataset row count and columns.


## 5) Create week × product_id revenue matrix

Pivot the weekly revenue table so each row is a year/week and each column is a product_id. The cell value is the total weekly revenue for that product.


In [40]:
# Convert year_week -> week-of-year integer (1..52)
trimmed_weekly_revenue["woy"] = (
    trimmed_weekly_revenue["year_week"].str.split("-").str[-1].astype(int)
)

# Keep only valid 1..52 weeks
trimmed_weekly_revenue = trimmed_weekly_revenue[
    trimmed_weekly_revenue["woy"].between(1, 52)
].copy()

# Rebuild week x product matrix using integer week index
weekly_product_matrix = trimmed_weekly_revenue.pivot_table(
    index="woy",
    columns="product_id",
    values="weekly_revenue",
    aggfunc="sum",
    fill_value=0,
).sort_index()

# Write both outputs
weekly_product_matrix_output = data_dir / "SP_2017_weekly_product_revenue_by_product_id.csv"
weekly_product_matrix.to_csv(
    weekly_product_matrix_output,
    index=True,
    index_label="woy",
)

print("Wrote week × product_id revenue matrix to:", weekly_product_matrix_output)
print("Matrix shape:", weekly_product_matrix.shape)

Wrote week × product_id revenue matrix to: ../data/SP_2017_weekly_product_revenue_by_product_id.csv
Matrix shape: (51, 2434)


In [41]:
type(weekly_product_matrix)
weekly_product_matrix.head()

product_id,0009406fd7479715e4bef61dd91f2462,00126f27c813603687e6ce486d909d01,001795ec6f1b187d37335e1c4704762e,001b72dfd63e9833e8c02742adf472e3,003c0b8f6580c850bd2e32044d2ac307,007c63ae4b346920756b5adcad8095de,00ba6d766f0b1d7b78a5ce3e1e033263,00bb62ea3729537a687c3fddcd123662,00df6fc5f33cc3f7ceec4ec6337d9cd7,0110573bc9195aa810a4384f189f48f5,...,ff28477a5bcb2fc34bb86a6c2eea1566,ff469d5a015226354ab92a85c8c46537,ff4c1a248a5110d784de5c67a9106d67,ff4f5b008c32f80c3f952b2b56a28e04,ff5d87897ed26d564711df324b98ebee,ff7fccf8513f360157f0660fe51d1d88,ff922797a6771cab4e0c51d482285ec3,ffa4ff381e25872b5bf85353ff7ab0c9,ffc48c754b5bd736e2887e279d1dec72,ffce5ed9e0bcc2e46796b988cdac733b
woy,,,,,,,,,,,,,,,,,,,,,
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
print("Weekly revenue rows:", len(weekly_sp_product_revenue))
print("Weekly revenue columns:", weekly_sp_product_revenue.columns.tolist())
weekly_sp_product_revenue.head()


Weekly revenue rows: 15371
Weekly revenue columns: ['year_week', 'product_id', 'product_category_name', 'weekly_revenue']


,year_week,product_id,product_category_name,weekly_revenue
0,2017-02,1a2d84c11fbc7b3d7c0a10206c085ce5,telefonia_fixa,10.49
1,2017-02,3889a02f55ae9a5e5e321c4a4d2eee64,moveis_decoracao,34.90
2,2017-02,743801b34cc44776de511ba8eff778e2,moveis_quarto,9.90
3,2017-02,a25a9e3433043fb8a0808f310d8d8203,fashion_bolsas_e_acessorios,129.99
4,2017-03,0ab80f38a21093b448518f68efe82c24,utilidades_domesticas,19.90
